# Run the bridges dataset builder (Kaggle or Colab)

This notebook only **calls** the code in `bridgebuild/`; all the logic lives in the Python files.

| Step | Command | Where to run |
|---|---|---|
| 1 Setup + HF token | – | any |
| 2 Dry run (local folder, 3 cells) | `build --local-dir` | any, 2 min |
| 3 Discover regions with bridges | `discover` | any |
| 4 Build → Hub (resumable, batched) | `build` | **Kaggle CPU** (32 GB RAM, 12 h) or Colab |
| 5 Finalize: split, leakage tests, manifest, card | `finalize` | any |
| 6 Analysis + overlay samples | `stats` | any |
| 7 Export YOLO-OBB folder + train | `to-yolo` | **Kaggle GPU** (P100 / 2×T4) or Colab T4 |

**Why it does not crash RAM:** one ~4 km cell at a time, imagery read one 256 px chip at a time, chips
flushed to a Parquet shard every 500 chips, each shard uploaded with the progress file and then deleted.
If the session dies, re-run step 4: finished cells are skipped (at most one shard of work is lost).

**Kaggle:** Settings → Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a *write* token).
**Colab:** 🔑 Secrets panel → `HF_TOKEN`, notebook access on.

## 1 · Setup

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
BASE = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.getcwd())
print("Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local", "| base:", BASE)

# Get the code: EITHER clone your repo (experiments/ folder) OR upload bridges-hf-builder.zip
REPO_URL = ""          # e.g. "https://github.com/<org>/<repo>.git"  (the builder sits in experiments/bridges-hf-builder)
CODE = f"{BASE}/bridges-hf-builder"
if not os.path.exists(CODE):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, f"{BASE}/repo"], check=True)
        CODE = f"{BASE}/repo/experiments/bridges-hf-builder"
    elif os.path.exists(f"{BASE}/bridges-hf-builder.zip"):
        subprocess.run(["unzip", "-q", f"{BASE}/bridges-hf-builder.zip", "-d", BASE], check=True)
    else:
        # Kaggle: attach the zip as a Dataset -> it appears under /kaggle/input/<name>/
        import glob
        z = glob.glob("/kaggle/input/*/bridges-hf-builder*.zip") + glob.glob("/kaggle/input/*/bridges-hf-builder")
        assert z, "upload bridges-hf-builder.zip or set REPO_URL"
        subprocess.run(["unzip", "-q", z[0], "-d", BASE], check=True) if z[0].endswith(".zip") else subprocess.run(["cp", "-r", z[0], BASE], check=True)
os.chdir(CODE); print("code:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
!python tests/test_units.py

In [ ]:
# HF token from the platform's secret store - never paste it in the notebook
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
elif ON_COLAB:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
me = whoami()["name"]; print("logged in as", me)

CONFIG = "configs/phase1_oam_france.yaml"
os.environ["BB_REPO_ID"] = f"{me}/fair-bridges-chips"     # or your org: "hotosm/fair-bridges-chips"
os.environ["PYTHONUNBUFFERED"] = "1"
print("dataset repo:", os.environ["BB_REPO_ID"])

## 2 · Dry run (writes to a local folder, nothing uploaded)
Checks that OAM, Overpass, IGN WFS/WMS answer from this machine. Look at the per-cell lines: bridges, images, chips, RAM.

In [ ]:
!python -m bridgebuild build --config $CONFIG --local-dir bb_dry_hub --max-cells 3
!python -m bridgebuild finalize --config $CONFIG --local-dir bb_dry_hub
!python -m bridgebuild stats --config $CONFIG --local-dir bb_dry_hub --overlays 12

In [ ]:
from IPython.display import Image as Show, display
display(Show("bb_work/reports/overlay_samples.png"))

## 3 · Discover where OAM imagery has bridges
Ranks open-licence OAM images in a search box by the number of OSM bridges inside. Copy the best ones into `regions:` of the config (step 4 only spends time where there are bridges).

In [ ]:
import sys; sys.path.insert(0, ".")
from bridgebuild.discover import discover, as_regions
import yaml
cand = discover((85.20, 27.60, 85.45, 27.80), max_gsd_m=0.6, out_csv="candidates_kathmandu.csv")   # W S E N
display(cand.head(15))
print(yaml.safe_dump({"regions": as_regions(cand, "kathmandu", top=5, country="NP", global_south=True)}, sort_keys=False))

## 4 · Build → Hugging Face (resumable)
Re-run this cell after any disconnect; it continues where it stopped. `--max-hours 11` stops cleanly before Kaggle's 12 h limit.
Use `--only oam_osm` / `--only france` to build one source per session (e.g. two Kaggle sessions in parallel would clash on the same repo — run them one after another).

In [ ]:
!python -m bridgebuild build --config $CONFIG --max-hours 11

In [ ]:
# Progress (from the Hub, no images downloaded)
import json
from huggingface_hub import hf_hub_download
st = json.load(open(hf_hub_download(os.environ["BB_REPO_ID"], "state/state.json", repo_type="dataset", force_download=True)))
print(st["totals"], "| failed cells:", len(st["failed_cells"]))
list(st["failed_cells"].items())[:10]

In [ ]:
# Optional: retry cells that failed because a service was down
!python -m bridgebuild build --config $CONFIG --retry-failed --max-hours 2

## 5 · Finalize: split + leakage tests + manifest + dataset card + record
Reads only the small index files. Fails (exit code 2) if any leakage test fails.

In [ ]:
!python -m bridgebuild finalize --config $CONFIG

## 6 · Analysis + overlay samples (pushed to `reports/` on the Hub)

In [ ]:
!python -m bridgebuild stats --config $CONFIG --push --overlays 24
import pandas as pd, glob
for f in ["chips_by_split", "bridges_by_source", "bridge_coverage", "licences"]:
    p = f"bb_work/reports/{f}.csv"
    if os.path.exists(p): print(f); display(pd.read_csv(p))
display(Show("bb_work/reports/overview.png")); display(Show("bb_work/reports/overlay_samples.png"))

In [ ]:
# Your own analysis: stream the dataset (no full download)
from datasets import load_dataset
ds = load_dataset(os.environ["BB_REPO_ID"], split="train", streaming=True)
ex = next(iter(ds)); print({k: v for k, v in ex.items() if k != "image"}); ex["image"]

## 7 · Export YOLO-OBB and train (GPU session)
Streams one shard at a time to `images/` + `labels/` + `data.yaml`. `--neg-keep 0.5` keeps half of the negative chips.

In [ ]:
OUT = f"{BASE}/yolo_bridges"
!python -m bridgebuild to-yolo --config $CONFIG --out $OUT --neg-keep 1.0
!du -sh $OUT; cat $OUT/data.yaml

In [ ]:
# Training with the doc 04 §6 settings. Needs a GPU session.
!pip install -q ultralytics
from ultralytics import YOLO
model = YOLO("yolo26s-obb.pt")          # DOTA weights; nc is reset to 1 from data.yaml
model.train(data=f"{OUT}/data.yaml", imgsz=256, epochs=100, batch=64, patience=20, close_mosaic=10,
            degrees=180, flipud=0.5, fliplr=0.5, scale=0.3, hsv_h=0.02, hsv_v=0.3, project=f"{BASE}/runs", name="phase1a")